# B2B BookStore Sales — Data Exploration

Loads `book_sales_practice.csv` into `raw_data` and takes a first look. `raw_data` stays untouched; cleaning will work on a copy.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

## Load the data

In [2]:
raw_data = pd.read_csv("book_sales_practice.csv")
print(f"Rows: {raw_data.shape[0]:,}  |  Columns: {raw_data.shape[1]}")
raw_data.head()

Rows: 19,004  |  Columns: 14


,InvoiceNo,InvoiceDate,CustomerID,CustomerName,CustomerType,City,Region,ProductID,BookTitle,Category,Publisher,Quantity,UnitPrice,DiscountRate
0,INV100016,2024-01-01,C1013,Atlas Kitap Evi,Bookstore,Izmir,Aegean,B061,Smart Marketing,Business,Cinar Publishing,14,355.0,0.34
1,INV100111,2024-01-01,C1098,Rota Halk Kutuphanesi Istanbul,Library,Istanbul,Marmara,B009,The Hidden Harbour,Fiction,Cinar Publishing,11,185.0,0.10
2,INV100111,2024-01-01,C1098,Rota Halk Kutuphanesi Istanbul,Library,Istanbul,Marmara,B041,Introduction to Chemistry,Science,Atlas Education,17,370.0,0.10
3,INV100111,2024-01-01,C1098,Rota Halk Kutuphanesi Istanbul,Library,Istanbul,Marmara,B029,Grade 12 Math Workbook,Education,Cinar Publishing,14,250.0,0.10
4,INV100111,2024-01-01,C1098,Rota Halk Kutuphanesi Istanbul,Library,Istanbul,Marmara,B028,Grade 6 Turkish Workbook,Education,Cinar Publishing,19,320.0,0.10


## Structure and data types

In [3]:
raw_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 19004 entries, 0 to 19003
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   InvoiceNo     19004 non-null  str    
 1   InvoiceDate   19004 non-null  str    
 2   CustomerID    19004 non-null  str    
 3   CustomerName  19004 non-null  str    
 4   CustomerType  18814 non-null  str    
 5   City          19004 non-null  str    
 6   Region        19004 non-null  str    
 7   ProductID     19004 non-null  str    
 8   BookTitle     19004 non-null  str    
 9   Category      19004 non-null  str    
 10  Publisher     19004 non-null  str    
 11  Quantity      19004 non-null  int64  
 12  UnitPrice     18851 non-null  float64
 13  DiscountRate  19004 non-null  float64
dtypes: float64(2), int64(1), str(11)
memory usage: 4.1 MB


In [4]:
raw_data.describe()

,Quantity,UnitPrice,DiscountRate
count,19004.000000,18851.000000,19004.000000
mean,41.887287,208.659222,0.240231
std,108.849543,68.946135,0.094165
min,-73.000000,80.000000,0.050000
25%,15.000000,165.000000,0.150000
50%,29.000000,190.000000,0.270000
75%,56.000000,255.000000,0.320000
max,13500.000000,375.000000,0.390000


## Raw data summary

**Size:** 19,004 rows × 14 columns, covering **2024-01-01 to 2025-12-31** (670 distinct dates).

### Data types
| Column(s) | Type when loaded | Note |
|---|---|---|
| `InvoiceDate` | `str` | Stored as text. Every value uses the same `YYYY-MM-DD` format and all of them parse, so it can be converted safely during cleaning. |
| `Quantity` | `int64` | OK |
| `UnitPrice`, `DiscountRate` | `float64` | OK |
| All other columns | `str` | OK |

### Missing values (nulls)
| Column | Missing rows | Can it be recovered? |
|---|---|---|
| `CustomerType` | 190 | Yes. Each customer always has the same type in its other rows. |
| `UnitPrice` | 153 | Yes. Each product always has the same price in its other rows. |

All other columns are complete.

### Typos and inconsistent text
- **`City`** has **24 spellings of 8 cities**, because of capitalisation and extra spaces. For example: `Istanbul`, `ISTANBUL`, `" istanbul "`. 380 values have leading or trailing spaces. Once the text is trimmed and title-cased, every city maps to exactly one region.
- **`CustomerType`, `Region`, `Category`, `Publisher`**: no typos. Each one has only clean, distinct values (4, 4, 7 and 5 values).
- **IDs and names match**: each `CustomerID` has one name and one type, and each `ProductID` has one title, category, publisher and price.

### Duplicates
- **40 rows are exact duplicates**, where the same invoice and product appear twice.

### Quantity
- **363 rows have a negative quantity** (as low as −73). These are **not errors**: all of them, and only them, have invoice numbers starting with `R` (e.g. `R100111`) rather than `INV`. They are **returns**.
- 6 return invoices are spread over more than one date, which is plausible for returns that were processed in parts.
- **3 extreme outliers**: quantities of 3,100, 3,100 and 13,500 (the median is 29, and only these 3 rows are above 500). These are probably data-entry errors.

### Value ranges (plausible)
- `UnitPrice`: 80 – 375
- `DiscountRate`: 0.05 – 0.39
- No zero quantities.

## Data quality checks

In [5]:
print("Missing values per column:")
print(raw_data.isna().sum())
print(f"\nDuplicate rows: {raw_data.duplicated().sum()}")
print(f"Date range: {raw_data['InvoiceDate'].min()} to {raw_data['InvoiceDate'].max()}")

Missing values per column:
InvoiceNo         0
InvoiceDate       0
CustomerID        0
CustomerName      0
CustomerType    190
City              0
Region            0
ProductID         0
BookTitle         0
Category          0
Publisher         0
Quantity          0
UnitPrice       153
DiscountRate      0
dtype: int64

Duplicate rows: 40
Date range: 2024-01-01 to 2025-12-31


In [6]:
# Number of unique values in each column
raw_data.nunique()

InvoiceNo       6262
InvoiceDate      670
CustomerID       120
CustomerName     120
CustomerType       4
City              24
Region             4
ProductID         74
BookTitle         74
Category           7
Publisher          5
Quantity         219
UnitPrice         41
DiscountRate      29
dtype: int64

# Data cleaning

All cleaning is done on `clean_data`, a copy, so `raw_data` stays unchanged.

In [7]:
clean_data = raw_data.copy()

## 1. Correct typos

`City` has 24 spellings of 8 cities (mixed capitals and extra spaces). Trim the spaces and use title case.

In [8]:
print("Before:", clean_data["City"].nunique(), "spellings")

clean_data["City"] = clean_data["City"].str.strip().str.title()

print("After: ", clean_data["City"].nunique(), "cities")
clean_data["City"].value_counts()

Before: 24 spellings
After:  8 cities


City
Istanbul     7926
Izmir        3243
Ankara       3240
Konya        1387
Bursa        1093
Adana         869
Antalya       761
Eskisehir     485
Name: count, dtype: int64

In [9]:
# Check: every city should now belong to exactly one region
clean_data.groupby("City")["Region"].nunique().max() == 1

np.True_

## 2. Fill missing values

Each customer always has the same `CustomerType`, and each product always has the same `UnitPrice`, in their other rows. So we fill each gap from the same customer's or product's other rows.

In [10]:
print("Missing before:")
print(clean_data[["CustomerType", "UnitPrice"]].isna().sum())

# Each ID has exactly one known value, so "first non-null value within the group" is that value
clean_data["CustomerType"] = clean_data.groupby("CustomerID")["CustomerType"].transform("first")
clean_data["UnitPrice"] = clean_data.groupby("ProductID")["UnitPrice"].transform("first")

print()
print("Missing after:")
print(clean_data.isna().sum().sum(), "missing cells in the whole table")

Missing before:
CustomerType    190
UnitPrice       153
dtype: int64

Missing after:
0 missing cells in the whole table


In [11]:
# Check: still exactly one type per customer and one price per product
(clean_data.groupby("CustomerID")["CustomerType"].nunique().max() == 1,
 clean_data.groupby("ProductID")["UnitPrice"].nunique().max() == 1)

(np.True_, np.True_)

## 3. Convert `InvoiceDate` to a date type

`format=` makes pandas reject any value that does not match `YYYY-MM-DD` instead of guessing.

In [12]:
print("Before:", clean_data["InvoiceDate"].dtype)

clean_data["InvoiceDate"] = pd.to_datetime(clean_data["InvoiceDate"], format="%Y-%m-%d")

print("After: ", clean_data["InvoiceDate"].dtype)
print("Range: ", clean_data["InvoiceDate"].min().date(), "to", clean_data["InvoiceDate"].max().date())

Before: str
After:  datetime64[us]
Range:  2024-01-01 to 2025-12-31


In [13]:
clean_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 19004 entries, 0 to 19003
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   InvoiceNo     19004 non-null  str           
 1   InvoiceDate   19004 non-null  datetime64[us]
 2   CustomerID    19004 non-null  str           
 3   CustomerName  19004 non-null  str           
 4   CustomerType  19004 non-null  str           
 5   City          19004 non-null  str           
 6   Region        19004 non-null  str           
 7   ProductID     19004 non-null  str           
 8   BookTitle     19004 non-null  str           
 9   Category      19004 non-null  str           
 10  Publisher     19004 non-null  str           
 11  Quantity      19004 non-null  int64         
 12  UnitPrice     19004 non-null  float64       
 13  DiscountRate  19004 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(10)
memory usage: 3.9 MB


## 4. Review duplicates (nothing deleted yet)

`keep=False` marks **every** copy, so each duplicated row is shown with its twin, sorted so pairs sit together.

In [14]:
dup_mask = clean_data.duplicated(keep=False)
duplicates = clean_data[dup_mask].sort_values(["InvoiceNo", "ProductID"])

print("Rows involved:", len(duplicates))
print("Extra copies that would be removed:", clean_data.duplicated().sum())
duplicates

Rows involved: 80
Extra copies that would be removed: 40


,InvoiceNo,InvoiceDate,CustomerID,CustomerName,CustomerType,City,Region,ProductID,BookTitle,Category,Publisher,Quantity,UnitPrice,DiscountRate
12,INV100094,2024-01-02,C1081,Defne Library Istanbul,Library,Istanbul,Marmara,B042,Introduction to Physics,Science,Pusula Press,11,280.0,0.14
15,INV100094,2024-01-02,C1081,Defne Library Istanbul,Library,Istanbul,Marmara,B042,Introduction to Physics,Science,Pusula Press,11,280.0,0.14
222,INV100124,2024-01-13,C1108,Lale Store,Online Retailer,Istanbul,Marmara,B045,The Little Harbour,History,Pusula Press,63,170.0,0.39
224,INV100124,2024-01-13,C1108,Lale Store,Online Retailer,Istanbul,Marmara,B045,The Little Harbour,History,Pusula Press,63,170.0,0.39
993,INV100399,2024-03-04,C1103,Okur Online,Online Retailer,Bursa,Marmara,B061,Smart Marketing,Business,Cinar Publishing,10,355.0,0.37
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18274,INV105875,2025-12-11,C1112,Rota Market,Online Retailer,Bursa,Marmara,B070,German A1 Workbook,Language Learning,Ada Yayinlari,76,215.0,0.34
8896,R102557,2025-01-03,C1063,Zeytin Koleji Istanbul,School,Istanbul,Marmara,B026,Grade 7 Math Workbook,Education,Atlas Education,-42,250.0,0.13
8897,R102557,2025-01-03,C1063,Zeytin Koleji Istanbul,School,Istanbul,Marmara,B026,Grade 7 Math Workbook,Education,Atlas Education,-42,250.0,0.13
19002,R105757,2025-12-31,C1057,Lale Anadolu Lisesi Istanbul,School,Istanbul,Marmara,B030,Grade 12 Chemistry Workbook,Education,Ada Yayinlari,-68,305.0,0.17


In [15]:
# Where do the duplicates come from?
duplicates.drop_duplicates()[["InvoiceNo", "InvoiceDate", "CustomerName", "BookTitle", "Quantity"]].assign(
    IsReturn=lambda d: d["InvoiceNo"].str.startswith("R")
)["IsReturn"].value_counts()

IsReturn
False    38
True      2
Name: count, dtype: int64

## 5. Review quantity outliers (nothing deleted yet)

Rows above 500 units. For comparison, each row is shown next to the **same customer's usual order size** for that book.

In [16]:
outliers = clean_data[clean_data["Quantity"] > 500].copy()

typical = (clean_data[clean_data["Quantity"].between(1, 500)]
           .groupby(["CustomerID", "ProductID"])["Quantity"].median()
           .rename("UsualQtyForThisCustomerAndBook"))
outliers = outliers.join(typical, on=["CustomerID", "ProductID"])
outliers["UsualQtyForThisCustomer"] = outliers["CustomerID"].map(
    clean_data[clean_data["Quantity"].between(1, 500)].groupby("CustomerID")["Quantity"].median())
outliers["GrossValue"] = outliers["Quantity"] * outliers["UnitPrice"]

print("Overall median quantity:", clean_data["Quantity"].median())
print("Largest quantity below 500:", clean_data.loc[clean_data["Quantity"] <= 500, "Quantity"].max())
outliers[["InvoiceNo", "InvoiceDate", "CustomerName", "CustomerType", "BookTitle",
          "Quantity", "UsualQtyForThisCustomerAndBook", "UsualQtyForThisCustomer", "UnitPrice", "GrossValue"]]

Overall median quantity: 29.0


Largest quantity below 500: 203


,InvoiceNo,InvoiceDate,CustomerName,CustomerType,BookTitle,Quantity,UsualQtyForThisCustomerAndBook,UsualQtyForThisCustomer,UnitPrice,GrossValue
4004,INV101146,2024-08-14,Rota Kitap Evi,Bookstore,The Silent Mountain,3100,24.5,23.0,140.0,434000.0
6241,INV101875,2024-10-11,Mimoza Kitabevi,Bookstore,The New Letter,3100,24.0,22.0,230.0,713000.0
10869,INV103263,2025-03-29,Lale Anadolu Lisesi Istanbul,School,Grade 8 Biology Workbook,13500,101.0,85.0,265.0,3577500.0


In [17]:
# The rest of each outlier's invoice, to see if the other lines look normal
clean_data[clean_data["InvoiceNo"].isin(outliers["InvoiceNo"])][
    ["InvoiceNo", "CustomerName", "BookTitle", "Quantity", "UnitPrice"]].sort_values("InvoiceNo")

,InvoiceNo,CustomerName,BookTitle,Quantity,UnitPrice
4002,INV101146,Rota Kitap Evi,Grade 5 Chemistry Workbook,31,180.0
4003,INV101146,Rota Kitap Evi,The Little Mountain,5,190.0
4004,INV101146,Rota Kitap Evi,The Silent Mountain,3100,140.0
4005,INV101146,Rota Kitap Evi,Simple Management,38,265.0
4006,INV101146,Rota Kitap Evi,The Little Garden,15,135.0
6241,INV101875,Mimoza Kitabevi,The New Letter,3100,230.0
6242,INV101875,Mimoza Kitabevi,The Open Bridge,33,280.0
6243,INV101875,Mimoza Kitabevi,The Hidden Forest,19,80.0
10867,INV103263,Lale Anadolu Lisesi Istanbul,Spanish A1 Workbook,70,225.0
10868,INV103263,Lale Anadolu Lisesi Istanbul,English A1 Workbook,46,140.0


In [18]:
# Distribution of the top quantities, to show the gap
clean_data["Quantity"].sort_values(ascending=False).head(15).tolist()

[13500, 3100, 3100, 203, 200, 200, 198, 198, 197, 190, 184, 183, 180, 177, 173]

## 6. Decision: remove duplicates and outliers

### Duplicates: delete
There are **40 duplicated rows**. In each pair, **every column has exactly the same value**: invoice number, date, customer, book, quantity, price and discount. A real order would not list the same book twice with identical details, so the second copy is a double entry. Keeping it would count those sales twice. We keep the first copy of each pair and delete the second.

### Outliers: delete
There are **3 rows** with quantities of **3,100, 3,100 and 13,500**. All other rows are 203 or less, and the median is 29.
- They are far larger than usual. **Lale Anadolu Lisesi Istanbul** is a school that normally orders about 85 books per line, and never more than 197. An order of 13,500 copies of one workbook is not realistic for a single school.
- The two bookstores (**Rota Kitap Evi** and **Mimoza Kitabevi**) usually order about 22–23 copies per line, and never more than 56. One line of 3,100 is more than 50 times their largest normal order.
- The other lines on the same invoices are normal (5–70 units). That points to **typing mistakes** (extra zeros, e.g. 31 → 3100) rather than real bulk orders.
- These 3 rows alone are about **2.8% of all gross sales**, so they would distort the totals, averages and charts in our analysis.

Because we can't know the correct quantities, we delete these rows instead of guessing a replacement.

**Note:** negative quantities are *not* outliers. They are returns (invoice numbers starting with `R`) and stay in the data.

In [19]:
rows_before = len(clean_data)

# 1. Remove duplicates: keep the first copy of each identical row
clean_data = clean_data.drop_duplicates(keep="first")
print(f"Duplicates removed: {rows_before - len(clean_data)}")

# 2. Remove outliers: quantities above 500 (the largest normal quantity is 203)
OUTLIER_LIMIT = 500
rows_before_outliers = len(clean_data)
clean_data = clean_data[clean_data["Quantity"] <= OUTLIER_LIMIT]
print(f"Outliers removed:   {rows_before_outliers - len(clean_data)}")

clean_data = clean_data.reset_index(drop=True)
print()
print(f"Rows: {rows_before:,} -> {len(clean_data):,}")

Duplicates removed: 40
Outliers removed:   3

Rows: 19,004 -> 18,961


In [20]:
# Check: no duplicates or outliers left, and the returns are still there
print("Duplicates left:", clean_data.duplicated().sum())
print("Max quantity:   ", clean_data["Quantity"].max())
print("Return rows:    ", (clean_data["Quantity"] < 0).sum())

Duplicates left: 0
Max quantity:    203
Return rows:     361


## 7. Working copy for analysis

`df` is a copy of `clean_data`, so analysis can change `df` without affecting the cleaned data.

In [21]:
df = clean_data.copy()
print(f"Rows: {df.shape[0]:,}  |  Columns: {df.shape[1]}")
df.head()

Rows: 18,961  |  Columns: 14


,InvoiceNo,InvoiceDate,CustomerID,CustomerName,CustomerType,City,Region,ProductID,BookTitle,Category,Publisher,Quantity,UnitPrice,DiscountRate
0,INV100016,2024-01-01,C1013,Atlas Kitap Evi,Bookstore,Izmir,Aegean,B061,Smart Marketing,Business,Cinar Publishing,14,355.0,0.34
1,INV100111,2024-01-01,C1098,Rota Halk Kutuphanesi Istanbul,Library,Istanbul,Marmara,B009,The Hidden Harbour,Fiction,Cinar Publishing,11,185.0,0.10
2,INV100111,2024-01-01,C1098,Rota Halk Kutuphanesi Istanbul,Library,Istanbul,Marmara,B041,Introduction to Chemistry,Science,Atlas Education,17,370.0,0.10
3,INV100111,2024-01-01,C1098,Rota Halk Kutuphanesi Istanbul,Library,Istanbul,Marmara,B029,Grade 12 Math Workbook,Education,Cinar Publishing,14,250.0,0.10
4,INV100111,2024-01-01,C1098,Rota Halk Kutuphanesi Istanbul,Library,Istanbul,Marmara,B028,Grade 6 Turkish Workbook,Education,Cinar Publishing,19,320.0,0.10


## 8. Add `IsReturn` and `NetSale` columns

- **`IsReturn`**: `True` when the invoice number starts with `R`. These are the return rows, and they have negative quantities.
- **`NetSale`**: the sale value after discount, `Quantity × UnitPrice × (1 − DiscountRate)`. For returns the quantity is negative, so `NetSale` is negative too. Summing `NetSale` therefore gives net sales with returns already subtracted.

In [22]:
df["IsReturn"] = df["InvoiceNo"].str.startswith("R")
df["NetSale"] = (df["Quantity"] * df["UnitPrice"] * (1 - df["DiscountRate"])).round(2)

# Check: every return has a negative quantity, and every sale a positive one
print("IsReturn matches negative quantity:", (df["IsReturn"] == (df["Quantity"] < 0)).all())
print(df["IsReturn"].value_counts())
print()
print(f"Sales:   {df.loc[~df['IsReturn'], 'NetSale'].sum():,.2f}")
print(f"Returns: {df.loc[df['IsReturn'], 'NetSale'].sum():,.2f}")
print(f"Net:     {df['NetSale'].sum():,.2f}")

df[["InvoiceNo", "Quantity", "UnitPrice", "DiscountRate", "IsReturn", "NetSale"]].head()

IsReturn matches negative quantity: True
IsReturn
False    18600
True       361
Name: count, dtype: int64



Sales:   130,915,951.00
Returns: -750,711.05
Net:     130,165,239.95


,InvoiceNo,Quantity,UnitPrice,DiscountRate,IsReturn,NetSale
0,INV100016,14,355.0,0.34,False,3280.2
1,INV100111,11,185.0,0.10,False,1831.5
2,INV100111,17,370.0,0.10,False,5661.0
3,INV100111,14,250.0,0.10,False,3150.0
4,INV100111,19,320.0,0.10,False,5472.0


## 9. Fact and dimension tables (star schema)

`df` is split into one **fact** table and four **dimension** tables:

| Table | One row per | Columns |
|---|---|---|
| `fact_sales` | invoice line (a sale or return of one book) | keys + `Quantity`, `UnitPrice`, `DiscountRate`, `IsReturn`, `NetSale` |
| `dim_customer` | customer | `CustomerID`, `CustomerName`, `CustomerType`, `City` |
| `dim_region` | city | `City`, `Region` |
| `dim_product` | book | `ProductID`, `BookTitle`, `Category`, `Publisher` |
| `dim_date` | calendar day | `Date`, `Year`, `Quarter`, `Month`, `MonthName`, `YearMonth`, `WeekDay` |

The fact table links to the dimensions through `CustomerID`, `ProductID` and `InvoiceDate` → `Date`. `dim_region` links to `dim_customer` through `City`, so each city's region is stored once instead of on every customer.

### Dimension tables

In [23]:
dim_customer = (df[["CustomerID", "CustomerName", "CustomerType", "City"]]
                .drop_duplicates()
                .sort_values("CustomerID")
                .reset_index(drop=True))

dim_region = (df[["City", "Region"]]
              .drop_duplicates()
              .sort_values(["Region", "City"])
              .reset_index(drop=True))

dim_product = (df[["ProductID", "BookTitle", "Category", "Publisher"]]
               .drop_duplicates()
               .sort_values("ProductID")
               .reset_index(drop=True))

# One row for every day in the period, including days with no sales
dates = pd.date_range(df["InvoiceDate"].min(), df["InvoiceDate"].max(), freq="D")
dim_date = pd.DataFrame({"Date": dates})
dim_date["Year"] = dim_date["Date"].dt.year
dim_date["Quarter"] = "Q" + dim_date["Date"].dt.quarter.astype(str)
dim_date["Month"] = dim_date["Date"].dt.month
dim_date["MonthName"] = dim_date["Date"].dt.month_name()
dim_date["YearMonth"] = dim_date["Date"].dt.strftime("%Y-%m")
dim_date["WeekDay"] = dim_date["Date"].dt.day_name()

print(f"dim_customer: {len(dim_customer):,} rows")
print(f"dim_region:   {len(dim_region):,} rows")
print(f"dim_product:  {len(dim_product):,} rows")
print(f"dim_date:     {len(dim_date):,} rows")

dim_customer: 120 rows
dim_region:   8 rows
dim_product:  74 rows
dim_date:     731 rows


In [24]:
dim_customer.head()

,CustomerID,CustomerName,CustomerType,City
0,C1001,Okur Kitap Evi,Bookstore,Ankara
1,C1002,Ada Kitap Evi,Bookstore,Izmir
2,C1003,Harf Kitabevi,Bookstore,Konya
3,C1004,Zeytin Kitabevi,Bookstore,Istanbul
4,C1005,Gunes Kitap Evi,Bookstore,Adana


In [25]:
dim_region

,City,Region
0,Izmir,Aegean
1,Ankara,Central Anatolia
2,Eskisehir,Central Anatolia
3,Konya,Central Anatolia
4,Bursa,Marmara
5,Istanbul,Marmara
6,Adana,Mediterranean
7,Antalya,Mediterranean


In [26]:
dim_product.head()

,ProductID,BookTitle,Category,Publisher
0,B001,The Open Road,Fiction,Cinar Publishing
1,B002,The Little Garden,Fiction,Ada Yayinlari
2,B003,The Little Mountain,Fiction,Ada Yayinlari
3,B004,The Silent Road,Fiction,Cinar Publishing
4,B005,The Little Island,Fiction,Cinar Publishing


In [27]:
dim_date.head()

,Date,Year,Quarter,Month,MonthName,YearMonth,WeekDay
0,2024-01-01,2024,Q1,1,January,2024-01,Monday
1,2024-01-02,2024,Q1,1,January,2024-01,Tuesday
2,2024-01-03,2024,Q1,1,January,2024-01,Wednesday
3,2024-01-04,2024,Q1,1,January,2024-01,Thursday
4,2024-01-05,2024,Q1,1,January,2024-01,Friday


### Fact table

In [28]:
fact_sales = df[["InvoiceNo", "InvoiceDate", "CustomerID", "ProductID",
                 "Quantity", "UnitPrice", "DiscountRate", "IsReturn", "NetSale"]].reset_index(drop=True)

print(f"fact_sales: {len(fact_sales):,} rows")
fact_sales.head()

fact_sales: 18,961 rows


,InvoiceNo,InvoiceDate,CustomerID,ProductID,Quantity,UnitPrice,DiscountRate,IsReturn,NetSale
0,INV100016,2024-01-01,C1013,B061,14,355.0,0.34,False,3280.2
1,INV100111,2024-01-01,C1098,B009,11,185.0,0.10,False,1831.5
2,INV100111,2024-01-01,C1098,B041,17,370.0,0.10,False,5661.0
3,INV100111,2024-01-01,C1098,B029,14,250.0,0.10,False,3150.0
4,INV100111,2024-01-01,C1098,B028,19,320.0,0.10,False,5472.0


### Checks

- Each dimension key is unique (one row per customer, book, day).
- Every key in the fact table exists in its dimension, and every customer's city exists in `dim_region`.
- The fact table has all rows of `df`, with the same total `NetSale`.

In [29]:
print("Unique keys:")
print("  CustomerID:", dim_customer["CustomerID"].is_unique)
print("  City:      ", dim_region["City"].is_unique)
print("  ProductID: ", dim_product["ProductID"].is_unique)
print("  Date:      ", dim_date["Date"].is_unique)

print("Every fact key found in its dimension:")
print("  Customers:", fact_sales["CustomerID"].isin(dim_customer["CustomerID"]).all())
print("  Cities:   ", dim_customer["City"].isin(dim_region["City"]).all())
print("  Products: ", fact_sales["ProductID"].isin(dim_product["ProductID"]).all())
print("  Dates:    ", fact_sales["InvoiceDate"].isin(dim_date["Date"]).all())

print("Same rows as df:    ", len(fact_sales) == len(df))
print("Same total NetSale: ", round(fact_sales["NetSale"].sum(), 2) == round(df["NetSale"].sum(), 2))

Unique keys:
  CustomerID: True
  City:       True
  ProductID:  True
  Date:       True
Every fact key found in its dimension:
  Customers: True
  Cities:    True
  Products:  True
  Dates:     True
Same rows as df:     True
Same total NetSale:  True


In [30]:
# Example: join the tables back together to answer a question - net sales by region and category
(fact_sales
 .merge(dim_customer, on="CustomerID")
 .merge(dim_region, on="City")
 .merge(dim_product, on="ProductID")
 .pivot_table(index="Region", columns="Category", values="NetSale", aggfunc="sum")
 .round(0))

Category,Business,Children,Education,Fiction,History,Language Learning,Science
Region,,,,,,,
Aegean,939901.0,2449273.0,12758922.0,2404139.0,1669327.0,5394635.0,4666203.0
Central Anatolia,2736479.0,3650620.0,11966344.0,4351080.0,3285464.0,5584961.0,5443297.0
Marmara,4909481.0,5369673.0,14135345.0,7268925.0,4407005.0,8143986.0,8436592.0
Mediterranean,568238.0,923107.0,3328643.0,1118143.0,868026.0,1581210.0,1806222.0


## 10. Save the clean data and the star schema to Excel

Everything goes into `book_sales_clean.xlsx`, one sheet per table:

| Sheet | Contents |
|---|---|
| `CleanData` | `df`: the cleaned data plus `IsReturn` and `NetSale` (first sheet, so `pd.read_excel` reads it by default) |
| `fact_sales` | fact table |
| `dim_customer`, `dim_region`, `dim_product`, `dim_date` | dimension tables |

Close the file in Excel before running this cell, otherwise saving fails with "Permission denied".

In [31]:
output_file = "book_sales_clean.xlsx"
sheets = {
    "CleanData": df,
    "fact_sales": fact_sales,
    "dim_customer": dim_customer,
    "dim_region": dim_region,
    "dim_product": dim_product,
    "dim_date": dim_date,
}

with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for name, table in sheets.items():
        table.to_excel(writer, sheet_name=name, index=False)

# Check: read every sheet back and compare with the table in memory
saved = pd.read_excel(output_file, sheet_name=None)
for name, table in sheets.items():
    same = saved[name].shape == table.shape and list(saved[name].columns) == list(table.columns)
    print(f"{name:13} {saved[name].shape[0]:>6,} rows x {saved[name].shape[1]:>2} columns   matches: {same}")

CleanData     18,961 rows x 16 columns   matches: True
fact_sales    18,961 rows x  9 columns   matches: True
dim_customer     120 rows x  4 columns   matches: True
dim_region         8 rows x  2 columns   matches: True
dim_product       74 rows x  4 columns   matches: True
dim_date         731 rows x  7 columns   matches: True
